# Inspect available data

In this notebook we will inspect data from the first 3 integrations:

- Day ahead prices from Nordpool
- Renewable generation from Renewables Ninja
- Imbalance prices from Energinet

## Import and read data

In [1]:
# Import libraries
from src.data import EnerginetAPI, NordpoolAPI, RenewablesNinjaAPI
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

## Nordpool

We integrated Nordpool day ahead prices. 

In [ ]:
# # Day ahead prices from Nordpool
nordpool_api = NordpoolAPI()

# # This endpoint queries, per day, so we will get the data for 2026 in a loop
# date_range = pd.date_range(start="2026-03-01", end="2026-03-26", freq="D")

# day_ahead_prices = []
# for date in date_range:
#     result = nordpool_api.day_ahead_price(date=date.strftime("%Y-%m-%d"), delivery_area="DK2")
#     time.sleep(0.5)  # to avoid hitting the API rate limit
#     day_ahead_prices.append(result)

# day_ahead_prices = pd.concat(day_ahead_prices).reset_index(drop=True)
# day_ahead_prices.to_csv("../data_samples/nordpool_day_ahead_prices_2026-03.csv", index=False)

In [6]:
day_ahead_prices = pd.read_csv("../data_samples/nordpool_day_ahead_prices_2026-03.csv", parse_dates=["deliveryStart", "deliveryEnd"])

In [7]:
day_ahead_prices

,deliveryStart,deliveryEnd,area,price_EUR
0,2026-02-28 23:00:00+00:00,2026-02-28 23:15:00+00:00,DK2,49.74
1,2026-02-28 23:15:00+00:00,2026-02-28 23:30:00+00:00,DK2,48.88
2,2026-02-28 23:30:00+00:00,2026-02-28 23:45:00+00:00,DK2,46.02
3,2026-02-28 23:45:00+00:00,2026-03-01 00:00:00+00:00,DK2,48.00
4,2026-03-01 00:00:00+00:00,2026-03-01 00:15:00+00:00,DK2,48.65
...,...,...,...,...
2491,2026-03-26 21:45:00+00:00,2026-03-26 22:00:00+00:00,DK2,120.74
2492,2026-03-26 22:00:00+00:00,2026-03-26 22:15:00+00:00,DK2,123.74
2493,2026-03-26 22:15:00+00:00,2026-03-26 22:30:00+00:00,DK2,114.03
2494,2026-03-26 22:30:00+00:00,2026-03-26 22:45:00+00:00,DK2,115.91


After analysing the requests I found out that it only allows to get data from this year (2026).

## Energinet

### Imbalance and day ahead prices

In [2]:
# # Imbalances
energinet_api = EnerginetAPI()
# imbalance_prices = energinet_api.imbalance_dataset(start="2024-01-01", end="2026-03-27", price_area="DK2")
# imbalance_prices.to_csv("../data_samples/energinet_imbalance_prices_2024-2026.csv", index=False)

In [4]:
imbalance_prices = pd.read_csv("../data_samples/energinet_imbalance_prices_2024-2026.csv")
imbalance_prices["TimeUTC"] = pd.to_datetime(imbalance_prices["TimeUTC"], utc=True)
imbalance_prices["TimeDK"] = pd.to_datetime(imbalance_prices["TimeDK"], utc=False) 

In [8]:
# Merge the two datasets on the time columns to compare prices
imbalance_prices[["TimeUTC", "SpotPriceEUR"]].merge(day_ahead_prices[["deliveryStart", "price_EUR"]], left_on="TimeUTC", right_on="deliveryStart", how="inner") 

,TimeUTC,SpotPriceEUR,deliveryStart,price_EUR
0,2026-02-28 23:00:00+00:00,49.74,2026-02-28 23:00:00+00:00,49.74
1,2026-02-28 23:15:00+00:00,48.88,2026-02-28 23:15:00+00:00,48.88
2,2026-02-28 23:30:00+00:00,46.02,2026-02-28 23:30:00+00:00,46.02
3,2026-02-28 23:45:00+00:00,48.00,2026-02-28 23:45:00+00:00,48.00
4,2026-03-01 00:00:00+00:00,48.65,2026-03-01 00:00:00+00:00,48.65
...,...,...,...,...
2491,2026-03-26 21:45:00+00:00,120.74,2026-03-26 21:45:00+00:00,120.74
2492,2026-03-26 22:00:00+00:00,123.74,2026-03-26 22:00:00+00:00,123.74
2493,2026-03-26 22:15:00+00:00,114.03,2026-03-26 22:15:00+00:00,114.03
2494,2026-03-26 22:30:00+00:00,115.91,2026-03-26 22:30:00+00:00,115.91


Energinet's Imbalance dataset also contains the spot price, and it has more data, so we will only use that dataset instead of Nordool's. 

### Wind generation and forecasts

A Useful resource is to have the actual wind generations and forecasts in a 5 min resolution given by energinet. There is also a dataset with the total capacity per technology that can be used.

The datasets are:
- Capacity per Municipality
- Electricity Production and Exchange 5 min Realtime
- Forecast Wind and Solar Power, 5 min

In [ ]:
# df_forecasts = energinet_api.forecasts_5min(start="2025-01-01", end="2026-03-27", price_area="DK2", forecast_type="Offshore Wind")
# df_forecasts.to_csv("../data_samples/energinet_forecasts_5min_offshore_wind_2025-2026.csv", index=False)

In [31]:
df_forecasts = pd.read_csv("../data_samples/energinet_forecasts_5min_offshore_wind_2025-2026.csv")
df_forecasts["Minutes5UTC"] = pd.to_datetime(df_forecasts["Minutes5UTC"], utc=True)
df_forecasts["TimestampUTC"] = pd.to_datetime(df_forecasts["TimestampUTC"], utc=True)
df_forecasts["Minutes5DK"] = pd.to_datetime(df_forecasts["Minutes5DK"], utc=False)
df_forecasts["TimestampDK"] = pd.to_datetime(df_forecasts["TimestampDK"], utc=False)
df_forecasts.head()

,Minutes5UTC,Minutes5DK,PriceArea,ForecastType,ForecastDayAhead,Forecast5Hour,Forecast1Hour,ForecastCurrent,TimestampUTC,TimestampDK
0,2026-03-26 22:55:00+00:00,2026-03-26 23:55:00,DK2,Offshore Wind,476.0,489.0,398.0,354.0,2026-03-26 22:55:27+00:00,2026-03-26 23:55:27
1,2026-03-26 22:50:00+00:00,2026-03-26 23:50:00,DK2,Offshore Wind,480.0,495.0,408.0,356.0,2026-03-26 22:55:27+00:00,2026-03-26 23:55:27
2,2026-03-26 22:45:00+00:00,2026-03-26 23:45:00,DK2,Offshore Wind,485.0,500.0,414.0,357.0,2026-03-26 22:45:29+00:00,2026-03-26 23:45:29
3,2026-03-26 22:40:00+00:00,2026-03-26 23:40:00,DK2,Offshore Wind,490.0,506.0,422.0,359.0,2026-03-26 22:45:29+00:00,2026-03-26 23:45:29
4,2026-03-26 22:35:00+00:00,2026-03-26 23:35:00,DK2,Offshore Wind,496.0,511.0,431.0,355.0,2026-03-26 22:40:27+00:00,2026-03-26 23:40:27


### Capacity per Municipality

In [ ]:
# df_capacity = energinet_api.capacity_per_municipality(start="2025-01-01", end="2026-03-01")
# # Munis from 101 to 400 are in DK2
# df_capacity['MunicipalityNo'] = df_capacity['MunicipalityNo'].astype(int)
# df_capacity_DK2 = df_capacity[df_capacity['MunicipalityNo']<=400].groupby('Month').sum()
# df_capacity_DK2.to_csv("../data_samples/energinet_capacity_per_municipality_DK2_2025-2026.csv")

In [23]:
df_capacity_DK2 = pd.read_csv("../data_samples/energinet_capacity_per_municipality_DK2_2025-2026.csv")
df_capacity_DK2["Month"] = pd.to_datetime(df_capacity_DK2["Month"])
df_capacity_DK2.head()

,Month,MunicipalityNo,CapacityGe100MW,CapacityLt100MW,OffshoreWindCapacity,OnshoreWindCapacity,SolarPowerCapacity,NumberGenerationUnitsGe100MW,NumberGenerationUnitsLt100MW,NumberOffshoreWindGenerators,NumberOnshoreWindGenerators,NumberSolarPanels
0,2025-01-01,11157,1645.0,928.778,1044.6,745.137,1049.66868,10,214,262,826,54156
1,2025-02-01,11157,1645.0,928.908,1044.6,745.137,1062.52460,10,218,262,826,54541
2,2025-03-01,11157,1653.0,928.908,1044.6,745.137,1065.78525,10,218,262,826,54854
3,2025-04-01,11157,1653.0,928.703,1044.6,745.137,1070.90153,10,213,262,826,55268
4,2025-04-30,11157,1653.0,927.732,1044.6,745.137,1075.71681,10,212,262,826,55723


Note that since 2025 to 2026 the offshore wind capacity in DK2 is constant 1044.6	

### Actual generation

In [ ]:
# df_production = energinet_api.production_exchange_5min(start="2025-01-01", end="2026-03-27", price_area="DK2")
# df_production.to_csv("../data_samples/energinet_production_exchange_5min_DK2_2025-2026.csv", index=False)

In [45]:
df_production = pd.read_csv("../data_samples/energinet_production_exchange_5min_DK2_2025-2026.csv")
df_production["Minutes5UTC"] = pd.to_datetime(df_production["Minutes5UTC"], utc=True)
df_production["Minutes5DK"] = pd.to_datetime(df_production["Minutes5DK"], utc=False)
df_production.head()

,Minutes5UTC,Minutes5DK,PriceArea,ProductionLt100MW,ProductionGe100MW,OffshoreWindPower,OnshoreWindPower,SolarPower,ExchangeGreatBelt,ExchangeGermany,ExchangeNetherlands,ExchangeGreatBritain,ExchangeNorway,ExchangeSweden,BornholmSE4
0,2026-03-26 22:55:00+00:00,2026-03-26 23:55:00,DK2,248.389999,331.829987,355.179993,172.220001,0.0,196.940002,-831.750000,NaN,NaN,NaN,1177.859985,10.21
1,2026-03-26 22:50:00+00:00,2026-03-26 23:50:00,DK2,253.979996,329.670013,358.850006,173.500000,0.0,190.380005,-836.260010,NaN,NaN,NaN,1212.390015,10.48
2,2026-03-26 22:45:00+00:00,2026-03-26 23:45:00,DK2,268.640015,329.190002,365.450012,178.229996,0.0,228.880005,-845.330017,NaN,NaN,NaN,1216.969971,10.44
3,2026-03-26 22:40:00+00:00,2026-03-26 23:40:00,DK2,273.600006,329.049988,367.000000,189.389999,0.0,215.199997,-856.169983,NaN,NaN,NaN,1242.750000,10.31
4,2026-03-26 22:35:00+00:00,2026-03-26 23:35:00,DK2,271.920013,329.160004,373.790009,195.009995,0.0,221.360001,-851.690002,NaN,NaN,NaN,1202.030029,11.09


## Renewables Ninja

In [54]:
# # Renewable generation
# reninja = RenewablesNinjaAPI()
# lon, lat = 12.933333, 55.016667  # Kriegers Flak location
# start = "2024-01-01"
# end = "2024-12-31"
# wind_data = reninja.wind_data(
#     lat=lat,
#     lon=lon,
#     date_from=start,
#     date_to=end,
#     capacity=500
# )
# wind_data.to_csv("../data_samples/renewables_ninja_wind_kriegers_flak_2024.csv")

In [55]:
wind_data = pd.read_csv("../data_samples/renewables_ninja_wind_kriegers_flak_2024.csv", parse_dates=["time"])
wind_data.head()

,time,electricity
0,2024-01-01 00:00:00,200.835
1,2024-01-01 01:00:00,176.968
2,2024-01-01 02:00:00,156.144
3,2024-01-01 03:00:00,168.183
4,2024-01-01 04:00:00,178.742


Renewable generation from Renewables Ninja is available only until 2024 and furthermore it is only available in hourly resolution.